---
### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 1.3 — Ground Truth Annotation

---

## 💬 Product Team Check-in

> **Sam:** "Hey — is there an easy way for users to provide feedback? Do they have to use the MLflow Server or can they send you something over Teams/Slack?."



**Objective:** We need to attach human ground-truth labels to those traces so we can:
- Measure how well automated judges match expert opinion
- Build alignment datasets for judge tuning
- Identify systematic failure modes before they reach production

In this notebook we'll:
1. Pull recent traces from the experiment
2. Log human feedback using `mlflow.log_feedback()`
3. Attach structured expectations with `mlflow.log_expectation()`
4. Inspect the annotated traces via the Python API

---
## 0 — Connect to MLflow

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()

# AI Client
openai_client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url=os.environ["GEMINI_OPENAI_BASE_URL"],
)
# Models
MODEL = "gemini-2.5-flash-lite"
EXPERIMENT_NAME = os.environ.get("EXPERIMENT_1_NAME", "mlflow-agent-1")
# Model Parameters
TEMPERATURE = 0.2
MAX_TOKENS = 512

# Same as before
SYSTEM_PROMPT = """You are an MLflow assistant."""

mlflow.openai.autolog()
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
experiment_id = experiment.experiment_id

---
## 1 — Generate some traces to annotate

We run a small batch of questions through the RAG agent. In practice you'd annotate traces already in the experiment — this cell just makes sure there's something to work with.

In [ ]:
# these should be different from the ones already there
sample_questions = [
    "How do I log a parameter in MLflow?",
    "What is the difference between mlflow.start_run() and mlflow.start_span()?",
    "How do I use RetrievalGroundedness in MLflow evaluation?",
    "Can MLflow track GPU metrics during training?",
    "How do I version prompts in MLflow 3?",
]

trace_ids = []

for question in sample_questions:
    openai_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        temperature=0.2,
    )
    trace_ids.append(mlflow.get_last_active_trace_id())
    print(f"  traced: {question[:70]}")

print(f"\nCollected {len(trace_ids)} trace IDs")

---
## 2 — Log feedback with `mlflow.log_feedback()`

`mlflow.log_feedback()` attaches a named assessment to a trace. The key parameters are:

| Parameter | Purpose |
|---|---|
| `trace_id` | Which trace to annotate |
| `name` | Assessment name — must match the judge name if used for alignment |
| `value` | The label (`"good"`, `"poor"`, a float score, etc.) |
| `rationale` | Optional explanation the alignment optimizer can learn from |
| `source` | Who/what provided the label (`HUMAN`, `LLM_JUDGE`, `CODE`) |

The `source` field is critical for alignment: MLflow uses it to distinguish human ground truth from automated judge outputs when computing agreement rates.

In [ ]:
from mlflow.entities import AssessmentSource, AssessmentSourceType
# Expert labels — one per question above.
# In production these come from the MLflow UI "Add Feedback" panel or a labeling tool.
human_labels = [
    ("good",       "Clear, code example included, correct API."),
    ("acceptable", "Distinction explained but misses start_span context manager usage."),
    ("poor",       "Described wrong scorer name. RetrievalGroundedness was not mentioned."),
    ("poor",       "MLflow does not natively track GPU metrics — response should say so."),
    ("good",       "Correctly described mlflow.genai.register_prompt() and load_prompt()."),
]

human_source = AssessmentSource(
    source_type=AssessmentSourceType.HUMAN,
    source_id="domain_expert",         # free-form identifier for the reviewer
)

for trace_id, (value, rationale), question in zip(trace_ids, human_labels, sample_questions):
    mlflow.log_feedback(
        trace_id=trace_id,
        name="response_quality",        # shared with the judge in experiment 4
        value=value,
        rationale=rationale,
        source=human_source,
    )
    print(f"  [{value:<10}]  {question[:65]}")

print(f"\nLogged {len(human_labels)} feedback assessments.")

---
## 3 — Log expectations with `mlflow.log_expectation()`

Feedback captures *how good* a response was. Expectations capture *what the correct answer should contain* — structured ground truth facts that scorers like `Correctness` compare against.

Both live on the trace and appear in the MLflow UI side-by-side with the actual outputs.

In [ ]:
expected_facts = [
    {"expected_facts": ["mlflow.log_param"]},
    {"expected_facts": ["start_run", "start_span", "run context", "span context manager"]},
    {"expected_facts": ["RetrievalGroundedness", "mlflow.genai.scorers"]},
    {"expected_facts": ["not supported natively", "plugin", "custom metric"]},
    {"expected_facts": ["mlflow.genai.register_prompt", "mlflow.genai.load_prompt", "aliases"]},
]

for trace_id, expectations in zip(trace_ids, expected_facts):
    mlflow.log_expectation(
        trace_id=trace_id,
        name="expectations",
        value=expectations,
    )

print(f"Logged expectations on {len(trace_ids)} traces.")

---
## 4 — Inspect annotated traces

We can retrieve any trace and read back its assessments (both feedback and expectations) via the Python API.

In [ ]:
# Pull back one trace and print its assessments
sample_trace = mlflow.get_trace(trace_ids[2])  # "RetrievalGroundedness" question

print(f"Trace ID : {sample_trace.info.trace_id}")
print(f"Question : {sample_questions[2]}")
print()

print("--- Assessments ---")
for assessment in sample_trace.info.assessments:
    print(f"  name      : {assessment.name}")
    print(f"  value     : {assessment.value}")
    if hasattr(assessment, 'rationale') and assessment.rationale:
        print(f"  rationale : {assessment.rationale}")
    print(f"  source    : {assessment.source.source_type}")
    print()

In [ ]:
sample_trace.info.assessments

---
## 5 — Search traces with feedback filters

`mlflow.search_traces()` supports filtering by assessment values. This lets you pull only the "poor" traces for review, or only traces where the human and judge disagree.

In [ ]:
# All traces in this experiment, returned as a list
all_traces = mlflow.search_traces(
    experiment_ids=[experiment_id],
    max_results=50,
    return_type="list",
)

print(f"Total traces in experiment : {len(all_traces)}")

# Summarise feedback distribution
from collections import Counter

label_counts = Counter()
for trace in all_traces:
    for assessment in trace.info.assessments:
        if assessment.name == "response_quality" and \
           assessment.source.source_type == AssessmentSourceType.HUMAN:
            label_counts[assessment.value] += 1

print("\nFeedback distribution (response_quality):")
for label, count in sorted(label_counts.items()):
    bar = "█" * count
    print(f"  {label:<12} {bar}  ({count})")

---
## Recap

| API | What it does |
|---|---|
| `mlflow.log_feedback(trace_id, name, value, rationale, source)` | Attach a human or automated label to a trace |
| `mlflow.log_expectation(trace_id, expectations)` | Attach structured ground truth facts to a trace |
| `mlflow.get_trace(trace_id)` | Retrieve a trace with all its assessments |
| `mlflow.search_traces(experiment_ids, ...)` | Query traces across an experiment |
| `AssessmentSource(source_type, source_id)` | Tag who provided the label — critical for alignment |

**What's next:** In Experiment 2 we'll pass these annotated traces to `judge.align()` with the SIMBA optimizer, teaching the judge to agree with the human labels above.